# Fine-tune NLU Courier — schema 7 kinds

Notebook tiếp tục fine-tune BamiBERT nhiều-head từ checkpoint cũ. Nó giữ 8 head (`goal`, `via`, `gkind`, `ganchor`, `vkind`, `vanchor`, `urgent`, `fragile`), nhưng `gkind/vkind` chỉ còn `none, near, far, north, south, east, west`. Weight của các lớp cũ được chuyển theo tên; các lớp `anchor_near` và `*_most` bị loại bỏ.

Notebook đọc toàn bộ `mission` có nhãn trong train/validation, bao gồm cả demo và query. NLP không cần ảnh. Không dùng test để fine-tune. Validation chỉ dùng đánh giá/chọn checkpoint.

Đầu vào: thư mục `delivery_public/{train,validation}/scenes.json` hoặc ZIP chứa hai file đó, cùng checkpoint `.pt` đã fine-tune trước. Sửa các đường dẫn ở CELL 1 rồi chạy từ đầu đến cuối. Checkpoint đầu ra dùng schema 7 kinds; inference cần dùng cùng schema và decoder ở CELL 6.


In [1]:
# CELL 1 — Config + load metadata từ FINAL1_data.zip trên Google Drive
%pip -q install 'transformers>=4.44,<5' sentencepiece accelerate

from pathlib import Path
import os

DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/FINAL1_data.zip')
try:
    from google.colab import drive
    if not DRIVE_ZIP_PATH.is_file():
        drive.mount('/content/drive')
except ImportError:
    pass

# Ngoài Colab: đặt FINAL1_data.zip cạnh notebook hoặc sửa đường dẫn này.
if not DRIVE_ZIP_PATH.is_file():
    LOCAL_ZIP_PATH = Path('./FINAL1_data.zip')
    if LOCAL_ZIP_PATH.is_file():
        DRIVE_ZIP_PATH = LOCAL_ZIP_PATH
DATA_ROOT = Path('./delivery_public')
if not DRIVE_ZIP_PATH.is_file() and not all(
    (DATA_ROOT / split / 'scenes.json').is_file() for split in ('train', 'validation')
):
    raise FileNotFoundError(
        f'Không thấy FINAL1_data.zip tại {DRIVE_ZIP_PATH} hoặc đủ scenes.json trong {DATA_ROOT}. '
        'Hãy upload ZIP lên Drive hoặc đặt delivery_public cạnh notebook.'
    )

# NLU chỉ cần mission text/labels trong scenes.json; không cần bung ảnh.
DATA_ZIP = DRIVE_ZIP_PATH
CHECKPOINT_PATH = Path('./bamibert_nlu_phase12_v3_4views_best.pt')
OUTPUT_DIR = Path('./nlu_courier_7kinds')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'Qualcomm-AI-Research/BamiBERT'
MAX_LEN = 256
EXTRA_POOLS = 3
SEED = 15102026
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
EPOCHS = 4
PATIENCE = 2
LR_ENCODER = 2e-6
LR_HEADS = 1.8e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

print('Data ZIP:', DATA_ZIP.resolve())
print('Train/validation scenes sẽ được đọc trực tiếp từ ZIP; ảnh không bị giải nén.')
print('Checkpoint NLU:', CHECKPOINT_PATH.resolve())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
Data ZIP: /content/drive/MyDrive/FINAL1_data.zip
Train/validation scenes sẽ được đọc trực tiếp từ ZIP; ảnh không bị giải nén.
Checkpoint NLU: /content/bamibert_nlu_phase12_v3_4views_best.pt


In [2]:
# CELL 2 — Imports, seed, exact output schema
import os, re, json, math, copy, random, zipfile, hashlib, unicodedata
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
KINDS = ['none','near','far','north','south','east','west']
HEADS = {
    'goal': PLACES + ['none'],
    'via': ['none'] + PLACES,
    'gkind': KINDS,
    'ganchor': ['none'] + PLACES,
    'vkind': KINDS,
    'vanchor': ['none'] + PLACES,
    'urgent': ['no','yes'],
    'fragile': ['no','yes'],
}
H2I = {h:{v:i for i,v in enumerate(labels)} for h,labels in HEADS.items()}
assert len(KINDS) == 7
print('device:', DEVICE, '| schema:', {h:len(v) for h,v in HEADS.items()})


device: cuda | schema: {'goal': 11, 'via': 11, 'gkind': 7, 'ganchor': 11, 'vkind': 7, 'vanchor': 11, 'urgent': 2, 'fragile': 2}


In [3]:
# CELL 3 — Đọc mission JSON trực tiếp từ FINAL1_data.zip (không giải nén ảnh)
def find_scenes(split):
    # Dùng FINAL1_data.zip làm nguồn chính, tránh vô tình đọc delivery_public cũ.
    if DATA_ZIP.is_file():
        suffix = f'{split}/scenes.json'
        with zipfile.ZipFile(DATA_ZIP) as zf:
            names = [n.replace('\\', '/') for n in zf.namelist()]
            matches = [n for n in names if n.endswith(suffix)]
            preferred = [n for n in matches if f'delivery_public/{suffix}' in n]
            matches = preferred or matches
            if not matches:
                raise FileNotFoundError(f'ZIP thiếu {suffix}: {DATA_ZIP}')
            shortest = min(len(n) for n in matches)
            best = [n for n in matches if len(n) == shortest]
            if len(best) != 1:
                raise ValueError(f'Nhiều file scenes.json phù hợp cho {split}: {best}')
            return json.loads(zf.read(best[0]).decode('utf-8'))
    direct = DATA_ROOT / split / 'scenes.json'
    if direct.is_file():
        return json.loads(direct.read_text(encoding='utf-8'))
    raise FileNotFoundError(f'Không thấy {DATA_ZIP} hoặc {direct}')

def make_row(scene, split):
    m = scene.get('mission') or {}
    text = str(m.get('text') or '').strip()
    if not text: raise ValueError(f"Mission text rỗng: {scene.get('scene_id')}")
    g = m.get('goal_ref') or {}; v = m.get('via_ref') or {}
    gkind = g.get('kind') or 'none'; vkind = v.get('kind') or 'none'
    ganchor = g.get('anchor') or 'none'; vanchor = v.get('anchor') or 'none'
    goal = m.get('goal'); via = m.get('via') or 'none'
    # Task mới giao tới một địa điểm có tên; tham chiếu chỉ định bản/vị trí của địa điểm đó.
    if goal not in PLACES: raise ValueError((scene.get('scene_id'),'goal',goal,m))
    if via not in PLACES + ['none']: raise ValueError((scene.get('scene_id'),'via',via,m))
    if gkind not in KINDS or vkind not in KINDS: raise ValueError((scene.get('scene_id'),'kind',gkind,vkind,m))
    if gkind in ('near','far'):
        if ganchor not in PLACES: raise ValueError((scene.get('scene_id'),'goal anchor',ganchor,m))
    elif ganchor != 'none': raise ValueError((scene.get('scene_id'),'unexpected goal anchor',gkind,ganchor,m))
    if via == 'none':
        if vkind != 'none' or vanchor != 'none': raise ValueError((scene.get('scene_id'),'waypoint without via',m))
    elif vkind in ('near','far'):
        if vanchor not in PLACES: raise ValueError((scene.get('scene_id'),'via anchor',vanchor,m))
    elif vanchor != 'none': raise ValueError((scene.get('scene_id'),'unexpected via anchor',vkind,vanchor,m))
    values = {'goal':goal,'via':via,'gkind':gkind,'ganchor':ganchor,
              'vkind':vkind,'vanchor':vanchor,
              'urgent':'yes' if bool(m.get('urgent',False)) else 'no',
              'fragile':'yes' if bool(m.get('fragile',False)) else 'no'}
    return {'text':text,'y':{h:H2I[h][values[h]] for h in HEADS},
            'values':values,'scene_id':scene.get('scene_id'),'split':split}

TRAIN_BASE = [make_row(s,'train') for s in TRAIN_SCENES]
VAL_BASE = [make_row(s,'validation') for s in VAL_SCENES]
assert len({r['scene_id'] for r in TRAIN_BASE}) == len(TRAIN_BASE)
assert len({r['scene_id'] for r in VAL_BASE}) == len(VAL_BASE)
print('train missions:',len(TRAIN_BASE),'| validation missions:',len(VAL_BASE))
for field in HEADS:
    print(field, Counter(r['values'][field] for r in TRAIN_BASE))


NameError: name 'TRAIN_SCENES' is not defined

In [ ]:
# CELL 4 — Safe text views (meaning-preserving)
_BRACKET = re.compile(r'\[[^\]]*\]')
def strip_accents(text):
    text = str(text).replace('đ','d').replace('Đ','D')
    return ''.join(c for c in unicodedata.normalize('NFD',text) if unicodedata.category(c) != 'Mn')
def clean_order_ids(text):
    return re.sub(r'\s+',' ',_BRACKET.sub(' ',text)).strip() or text
def four_views(text):
    base = clean_order_ids(text)
    return [text, base, strip_accents(text), strip_accents(base)]

def expand_views(rows):
    return [{'text':view,'y':row['y'],'base_id':row['scene_id'],'view':vi}
            for row in rows for vi,view in enumerate(four_views(row['text']))]

TRAIN_ROWS = expand_views(TRAIN_BASE)
VAL_ROWS = expand_views(VAL_BASE)
# Không thêm route/ảnh hoặc test vào NLU; mỗi view giữ nguyên nhãn semantic gốc.
print('train text views:',len(TRAIN_ROWS),'| validation text views:',len(VAL_ROWS))
print('sample views:')
for t in four_views(TRAIN_BASE[0]['text']): print(' •',t[:180])


In [ ]:
# CELL 5 — Model architecture + checkpoint transfer by label name
class MultiHeadNLU(nn.Module):
    def __init__(self, model_name=MODEL_NAME, extra_pools=EXTRA_POOLS):
        super().__init__()
        self.enc = AutoModel.from_pretrained(model_name)
        hidden = self.enc.config.hidden_size
        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)
        self.pool_q = nn.ModuleDict({h:nn.Sequential(nn.Linear(hidden,hidden//2),nn.Tanh(),nn.Linear(hidden//2,1)) for h in HEADS})
        self.heads = nn.ModuleDict({h:nn.Linear(hidden*2,len(labels)) for h,labels in HEADS.items()})
        self.extra_pool_q = nn.ModuleDict({h:nn.Linear(hidden,extra_pools,bias=False) for h in HEADS})
        extra_size = hidden*(extra_pools+1)
        self.extra_norm = nn.ModuleDict({h:nn.LayerNorm(extra_size) for h in HEADS})
        self.extra_heads = nn.ModuleDict({h:nn.Linear(extra_size,len(labels)) for h,labels in HEADS.items()})
        for h in HEADS:
            nn.init.zeros_(self.pool_q[h][2].weight); nn.init.zeros_(self.pool_q[h][2].bias)
            nn.init.normal_(self.extra_pool_q[h].weight,mean=0.0,std=0.02)
            nn.init.zeros_(self.extra_heads[h].weight); nn.init.zeros_(self.extra_heads[h].bias)

    def forward(self, ids, mask):
        hidden = self.enc(input_ids=ids,attention_mask=mask).last_hidden_state
        cls = hidden[:,0,:]; out = {}; token_mask = mask.unsqueeze(-1)==0
        for h in HEADS:
            score = self.pool_q[h](hidden).squeeze(-1).float().masked_fill(mask==0,-1e4)
            weight = score.softmax(-1).to(hidden.dtype)
            pooled = (weight.unsqueeze(-1)*hidden).sum(1)
            old_logits = self.heads[h](self.drop(torch.cat([cls,pooled],-1)))
            ex_score = self.extra_pool_q[h](hidden).float().masked_fill(token_mask,-1e4)
            ex_weight = ex_score.softmax(1).to(hidden.dtype)
            ex_pool = torch.einsum('btk,bth->bkh',ex_weight,hidden).flatten(1)
            feature = self.extra_norm[h](torch.cat([cls,ex_pool],-1))
            out[h] = old_logits + self.extra_heads[h](self.drop(feature))
        return out

def load_and_remap_checkpoint(path, model):
    ckpt = torch.load(path,map_location='cpu',weights_only=False)
    if not isinstance(ckpt,dict) or 'state' not in ckpt or 'heads' not in ckpt:
        raise ValueError('Checkpoint cần dict có keys state và heads.')
    old_heads = ckpt['heads']; old_state = ckpt['state']
    if set(old_heads) != set(HEADS): raise ValueError(f'Checkpoint head mismatch: {list(old_heads)}')
    for h in HEADS:
        missing = set(HEADS[h])-set(old_heads[h])
        if missing: raise ValueError(f'{h}: checkpoint thiếu nhãn dùng chung {missing}')
    new_state = model.state_dict()
    remapped = []
    for key,target in new_state.items():
        source = old_state.get(key)
        if source is None:
            raise KeyError(f'Checkpoint thiếu tensor {key}')
        if key in ('heads.gkind.weight','heads.gkind.bias','extra_heads.gkind.weight','extra_heads.gkind.bias',
                   'heads.vkind.weight','heads.vkind.bias','extra_heads.vkind.weight','extra_heads.vkind.bias'):
            h = key.split('.')[1]
            for new_i,label in enumerate(HEADS[h]):
                old_i = old_heads[h].index(label)
                target[new_i].copy_(source[old_i])
            remapped.append((h,list(old_heads[h]),list(HEADS[h])))
        elif tuple(source.shape) == tuple(target.shape):
            target.copy_(source)
        else:
            raise ValueError(f'Shape mismatch {key}: checkpoint {tuple(source.shape)} vs model {tuple(target.shape)}')
    model.load_state_dict(new_state,strict=True)
    print('Loaded source phase:',ckpt.get('phase'),'| epoch:',ckpt.get('epoch'))
    print('Kind head remap:',remapped)
    print('Dropped labels:',{h:[x for x in old_heads[h] if x not in HEADS[h]] for h in ('gkind','vkind')})
    return ckpt

if not CHECKPOINT_PATH.is_file(): raise FileNotFoundError(f'Checkpoint không thấy: {CHECKPOINT_PATH}')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = MultiHeadNLU().to(DEVICE)
SOURCE_CKPT = load_and_remap_checkpoint(CHECKPOINT_PATH,model)
print('Model ready; trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# CELL 6 — Constrained decoder + metrics
def log_probs(logits):
    return {h:F.log_softmax(v.float(),dim=-1).cpu().numpy() for h,v in logits.items()}

def best_ref(kind_lp,anchor_lp):
    best=(-1e30,'none','none')
    for kind in KINDS:
        anchors = PLACES if kind in ('near','far') else ['none']
        for anchor in anchors:
            score=float(kind_lp[H2I['gkind'][kind]]+anchor_lp[H2I['ganchor'][anchor]])
            if score>best[0]: best=(score,kind,anchor)
    return best

def constrained_decode(lp):
    n=len(lp['goal']); preds=[]
    for i in range(n):
        # Goal is always a named campus place in this task; its spatial reference selects a copy/location.
        gbest=None
        for goal in PLACES:
            ref_score,gkind,ganchor=best_ref(lp['gkind'][i],lp['ganchor'][i])
            score=float(lp['goal'][i,H2I['goal'][goal]]+ref_score)
            cand=(score,goal,gkind,ganchor)
            if gbest is None or score>gbest[0]: gbest=cand
        # Via may be absent; when absent its reference heads must also be none.
        vnone=(float(lp['via'][i,H2I['via']['none']]+lp['vkind'][i,H2I['vkind']['none']]+lp['vanchor'][i,H2I['vanchor']['none']]),'none','none','none')
        vbest=vnone
        for via in PLACES:
            kind_lp=lp['vkind'][i]; anchor_lp=lp['vanchor'][i]
            ref_best=(-1e30,'none','none')
            for kind in KINDS:
                anchors=PLACES if kind in ('near','far') else ['none']
                for anchor in anchors:
                    score=float(kind_lp[H2I['vkind'][kind]]+anchor_lp[H2I['vanchor'][anchor]])
                    if score>ref_best[0]: ref_best=(score,kind,anchor)
            score=float(lp['via'][i,H2I['via'][via]]+ref_best[0])
            if score>vbest[0]: vbest=(score,via,ref_best[1],ref_best[2])
        attrs={h:HEADS[h][int(lp[h][i].argmax())] for h in ('urgent','fragile')}
        preds.append({'goal':gbest[1],'gkind':gbest[2],'ganchor':gbest[3],
                      'via':vbest[1],'vkind':vbest[2],'vanchor':vbest[3],**attrs})
    return preds

@torch.inference_mode()
def predict_rows(rows,batch_size=EVAL_BATCH_SIZE):
    model.eval(); outs=[]
    for start in range(0,len(rows),batch_size):
        chunk=rows[start:start+batch_size]
        enc=tok([r['text'] for r in chunk],padding=True,truncation=True,max_length=MAX_LEN,return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
            logits=model(enc['input_ids'],enc['attention_mask'])
        lp=log_probs(logits); outs.extend(constrained_decode(lp))
    return outs

def evaluate(rows,base_rows):
    pred=predict_rows(rows); exact=Counter(); total=Counter(); per_head=Counter(); n=len(rows)
    for r,p in zip(rows,pred):
        truth={h:HEADS[h][r['y'][h]] for h in HEADS}
        view=r['view']; total[view]+=1
        exact[view]+=int(all(p[h]==truth[h] for h in HEADS))
        for h in HEADS: per_head[(view,h)]+=int(p[h]==truth[h])
    result={v:exact[v]/max(1,total[v]) for v in total}
    print('Exact 8-head by view:',{v:round(result[v],4) for v in sorted(result)})
    for view in sorted(total):
        print(view,'head acc:',{h:round(per_head[(view,h)]/total[view],4) for h in HEADS})
    return float(np.mean(list(result.values()))),result,pred

BASELINE_SCORE,BASELINE_BY_VIEW,VAL_PRED=evaluate(VAL_ROWS,VAL_BASE)
print('Baseline mean exact8:',round(BASELINE_SCORE,5))


In [ ]:
# CELL 7 — Low-LR fine-tune; baseline checkpoint is retained unless validation improves
class MissionDataset(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.rows[i]

def collate(batch):
    enc=tok([r['text'] for r in batch],padding=True,truncation=True,max_length=MAX_LEN,return_tensors='pt')
    y={h:torch.tensor([r['y'][h] for r in batch],dtype=torch.long) for h in HEADS}
    return enc,y

train_loader=DataLoader(MissionDataset(TRAIN_ROWS),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,
                        num_workers=0,pin_memory=USE_AMP)
enc_params=[p for n,p in model.named_parameters() if n.startswith('enc.')]
head_params=[p for n,p in model.named_parameters() if not n.startswith('enc.')]
optimizer=torch.optim.AdamW([{'params':enc_params,'lr':LR_ENCODER},{'params':head_params,'lr':LR_HEADS}],weight_decay=WEIGHT_DECAY)
total_steps=max(1,len(train_loader)*EPOCHS)
scheduler=get_linear_schedule_with_warmup(optimizer,int(0.08*total_steps),total_steps)
scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda' and AMP_DTYPE==torch.float16))
LOSS_W={'goal':2.0,'via':2.0,'gkind':1.5,'ganchor':1.3,'vkind':1.1,'vanchor':1.1,'urgent':1.0,'fragile':1.0}

def multitask_loss(logits,y):
    return sum(LOSS_W[h]*F.cross_entropy(logits[h].float(),y[h].to(DEVICE),label_smoothing=0.01) for h in HEADS)/sum(LOSS_W.values())

def save_ckpt(path,epoch,score,by_view):
    torch.save({'phase':13,'schema':'courier_7_kinds_v1','epoch':epoch,'heads':copy.deepcopy(HEADS),
                'state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'model_name':MODEL_NAME,'max_len':MAX_LEN,'extra_pools':EXTRA_POOLS,
                'source_checkpoint':str(CHECKPOINT_PATH),'val_exact8':score,'val_by_view':by_view,
                'config':{'lr_encoder':LR_ENCODER,'lr_heads':LR_HEADS,'epochs':EPOCHS,'seed':SEED}},path)

BEST_PATH=OUTPUT_DIR/'bamibert_nlu_courier_7kinds_best.pt'
REPORT_PATH=OUTPUT_DIR/'finetune_report.json'
# Epoch 0 is a valid fallback if fine-tuning hurts the previous model.
save_ckpt(BEST_PATH,0,BASELINE_SCORE,BASELINE_BY_VIEW)
best=BASELINE_SCORE; best_epoch=0; bad_epochs=0; history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); running=0.0
    for enc,y in tqdm(train_loader,desc=f'epoch {epoch}/{EPOCHS}'):
        ids=enc['input_ids'].to(DEVICE); mask=enc['attention_mask'].to(DEVICE)
        y={h:t.to(DEVICE) for h,t in y.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
            logits=model(ids,mask); loss=multitask_loss(logits,y)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running+=float(loss.detach())
    VAL_SCORE,VAL_BY_VIEW,VAL_PRED=evaluate(VAL_ROWS,VAL_BASE)
    improved=VAL_SCORE>best+1e-5
    history.append({'epoch':epoch,'train_loss':running/max(1,len(train_loader)),
                    'val_exact8':VAL_SCORE,'val_by_view':VAL_BY_VIEW,'improved':improved})
    print(f"epoch={epoch} train_loss={running/max(1,len(train_loader)):.4f} val_exact8={VAL_SCORE:.5f} best={best:.5f}")
    if improved:
        best=VAL_SCORE; best_epoch=epoch; bad_epochs=0
        save_ckpt(BEST_PATH,epoch,best,VAL_BY_VIEW)
    else:
        bad_epochs+=1
        if bad_epochs>=PATIENCE:
            print('Early stopping; giữ checkpoint có validation cao nhất.')
            break

# Reload best and persist an auditable summary.
final_ckpt=torch.load(BEST_PATH,map_location='cpu',weights_only=False)
model.load_state_dict(final_ckpt['state'],strict=True); model.to(DEVICE).eval()
FINAL_SCORE,FINAL_BY_VIEW,FINAL_PRED=evaluate(VAL_ROWS,VAL_BASE)
report={'best_checkpoint':str(BEST_PATH),'source_checkpoint':str(CHECKPOINT_PATH),
        'schema':HEADS,'train_missions':len(TRAIN_BASE),'validation_missions':len(VAL_BASE),
        'train_views':len(TRAIN_ROWS),'validation_views':len(VAL_ROWS),
        'baseline_exact8':BASELINE_SCORE,'best_exact8':FINAL_SCORE,'best_epoch':final_ckpt['epoch'],
        'baseline_by_view':BASELINE_BY_VIEW,'best_by_view':FINAL_BY_VIEW,'history':history}
REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',BEST_PATH.resolve()); print('Report:',REPORT_PATH.resolve())


In [ ]:
# CELL 8 — Final sanity checks and inference payload example
assert final_ckpt['heads']==HEADS
assert len(HEADS['gkind'])==7 and len(HEADS['vkind'])==7
assert all(p['goal'] in PLACES for p in FINAL_PRED)
assert all((p['via']=='none') == (p['vkind']=='none') for p in FINAL_PRED)
print('Schema and constrained decoder checks: OK')
print('Sample parsed mission:',FINAL_PRED[0])
print('For inference, update the runtime HEADS/decoder to this saved 7-kind schema; the old 12-kind runtime will reject this checkpoint.')
